# Procena broja stanovnika iz otisaka zgrada (pristup 2)

Izgrađenost je direktan signal za broj stanovnika: gde ima više i gušće zgrada, po pravilu
živi više ljudi. Zato ovde gledamo otiske zgrada umesto sirovog snimka. Otiske
rasterizujemo po naselju u 2 kanala (pokrivenost = udeo ćelije pod zgradom, gustina zgrada
= broj centroida po ćeliji) i fine-tune-ujemo ResNet-18 (ImageNet) na cilj
`log1p(broj_stanovnika)`. Podela po opštinama (GroupKFold), praćenje kroz MLflow.

## Instalacija

In [ ]:
%pip install -q timm "mlflow>=3.0"

## Konfiguracija

In [ ]:
import os
import sys
# koren repoa na sys.path (penji se od radnog dir dok ne nadje core/)
koren = os.getcwd()
while not os.path.isdir(os.path.join(koren, "core")) and os.path.dirname(koren) != koren:
    koren = os.path.dirname(koren)
sys.path.insert(0, koren)

import glob

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
import mlflow
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

from core import (
    seed_everything, run_pass, two_phase_train,
    make_loaders,
    make_folds, oof_metrics, run_metrics, summary_line,
    cv_summary_figure, GLAVNE_KOLONE,
    setup_mlflow, output_dir, save_oof,
)
from scripts import config     # putanje do podataka i spisak strukturiranih atributa

OUT_DIR = output_dir()   # tezine modela i OOF parquet (UC Volume, lokalno "out/")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CFG = {
    "num_bands": 2,                 # 2 kanala: pokrivenost, gustina zgrada
    "img_px": 224,
    "epochs_head": 3,
    "epochs_finetune": 50,
    "batch_size": 64,
    "head_lr": 1e-3,
    "finetune_lr": 3e-4,
    "seed": 42,
}
seed_everything(CFG["seed"])
print("device:", DEVICE, "| footprint cutouts:", len(os.listdir(config.FOOTPRINT_CUT)))

## Podaci i podela

In [ ]:
labele = pd.read_parquet(config.NASELJE_TABLE)[
    ["naselje_maticni_broj", "opstina_maticni_broj", "pop"]]
df = pd.DataFrame({"path": glob.glob(os.path.join(config.FOOTPRINT_CUT, "*.npy"))})
df["naselje_maticni_broj"] = df.path.map(lambda f: int(os.path.splitext(os.path.basename(f))[0]))
df = df.merge(labele, on="naselje_maticni_broj", how="inner")
df["y"] = np.log1p(df["pop"]).astype("float32")

FOLDS = make_folds(df)
N_FOLDS = len(FOLDS)
broj_opstina = df["opstina_maticni_broj"].nunique()
print(f"uzoraka {len(df)} | opstina {broj_opstina} | foldova {N_FOLDS}")
for i, (t, v) in enumerate(FOLDS):
    print(f"  fold {i}: trening {len(t)} / val {len(v)} naselja ({v['opstina_maticni_broj'].nunique()} opstina)")

## Model

In [ ]:
# timm prilagodjava conv1 na 2 kanala (in_chans); model se pravi po foldu u train_fold
loss_fn = nn.HuberLoss()
use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

## Trening (sa MLflow praćenjem)

In [ ]:
def train_fold(train_frame, val_frame):
    # Istrenira jedan fold i vrati (best_state, best_val_r2, oof_pred_pop, net).
    train_dl, val_dl = make_loaders(train_frame, val_frame, CFG["batch_size"], CFG["seed"])
    net = timm.create_model("resnet18", pretrained=True, in_chans=CFG["num_bands"], num_classes=1).to(DEVICE)

    def epoch(opt, korak, freeze_bn=False):
        tl, _, _ = run_pass(net, train_dl, True, loss_fn, opt, scaler=scaler,
                          use_amp=use_amp, device=DEVICE, freeze_bn=freeze_bn)
        vl, P, Y = run_pass(net, val_dl, False, loss_fn, device=DEVICE)
        r2 = r2_score(Y, P)
        mlflow.log_metrics({"train_loss": tl, "val_loss": vl, "val_r2": r2}, step=korak)
        return r2

    best_r2, best_state = two_phase_train(
        net, epoch,
        CFG["epochs_head"], CFG["epochs_finetune"],
        CFG["head_lr"],     CFG["finetune_lr"],
    )
    net.load_state_dict(best_state)
    _, P, _ = run_pass(net, val_dl, False, loss_fn, device=DEVICE)
    oof_pred_pop = np.expm1(P)          # OOF predikcija u populaciji
    return best_state, best_r2, oof_pred_pop, net

## Evaluacija

In [ ]:
def run():
    # Puna k-struka GroupKFold CV (footprint pristup).
    setup_mlflow()   # Databricks workspace; lokalno mlruns/, ili Databricks preko env varijabli
    oof = pd.Series(np.nan, index=df.naselje_maticni_broj.values, dtype="float32")
    fold_r2 = []

    with mlflow.start_run(run_name=f"footprint-cv{len(FOLDS)}"):
        mlflow.log_params(CFG)
        mlflow.log_params({
            "pristup": "footprint",
            "backbone": "resnet18",
            "pretrained": True,
            "optimizer": "AdamW",
            "scheduler": "CosineAnnealingLR",
            "loss": "HuberLoss",
            "cv": f"GroupKFold(opstina) x{len(FOLDS)}",
            "n_uzoraka": len(df),
            "n_opstina": int(broj_opstina),
        })

        for fold, (train_frame, val_frame) in enumerate(FOLDS):
            with mlflow.start_run(run_name=f"footprint-fold{fold}", nested=True):
                mlflow.log_params({
                    **CFG,
                    "fold": fold,
                    "n_train": len(train_frame),
                    "n_val": len(val_frame)
                })
                best_state, best_r2, oof_pred_pop, net = train_fold(train_frame, val_frame)
                mlflow.log_metric("best_val_r2", best_r2)
                oof.loc[val_frame.naselje_maticni_broj.values] = oof_pred_pop
                put = f"{OUT_DIR}/footprint_fold{fold}.pt"
                torch.save(best_state, put)
                mlflow.log_artifact(put)
                mlflow.pytorch.log_model(
                    net,
                    name=f"model_footprint_fold{fold}",
                    serialization_format="pickle"
                )
                fold_r2.append(best_r2)
                print(f"[fold {fold}] best val R2 {best_r2:.3f}")

        oof_pred = oof.loc[df.naselje_maticni_broj.values].values.astype("float32")
        stvarno  = df["pop"].values.astype("float32")
        agg = run_metrics(fold_r2, stvarno, oof_pred, df)
        mlflow.log_metrics(agg)
        put_oof = save_oof(df, oof_pred, "footprint", OUT_DIR)   # ulaz za fuziju
        mlflow.log_artifact(put_oof)
        fig = cv_summary_figure(fold_r2, agg, stvarno, oof_pred, df, label="footprint")
        plt.show()
        mlflow.log_figure(fig, "cv_evaluacija_footprint.png")

    print(summary_line(agg, "footprint"))
    return {"pristup": "footprint", **agg}


rezultat = run()
print("\n=== Rezultat (footprint) ===")
display(pd.DataFrame([rezultat]).set_index("pristup"))

## Pristup 2a: strukturirani atributi (tabelarni MLP)

Isti otisci nose i signal koji se može prebrojati (koliko zgrada, kolika površina, kako su
raspoređene), pa ne mora sve da ide kroz piksele. Zato ovde otiske koristimo kao
strukturirane podatke: po naselju 14 atributa koje računa
`scripts/footprint/per_naselje.py`, bez ijedne slike. Uz to, pokriva sva naselja, ne samo
ona sa satelitskim isečcima, i ne traži GPU.

Tri grupe atributa: količina (broj zgrada, ukupna krovna površina), oblik (prosečna,
medijalna, p90 i rasipanje veličine, kompaktnost, udeo zgrada preko 200 m2) i raspored
(rastojanje do najbliže zgrade, broj suseda u 50 m). Raspored razdvaja zbijeno selo od
rastrkanog sa istim brojem zgrada, što sam broj zgrada ne vidi; udeo velikih hvata
stambene blokove.

Svi atributi su izvedeni iz geometrije. Overture opisne kolone ne koristimo: `num_floors`
je popunjen u 0.76% zgrada, `height` u 0.05%, `subtype`/`class` u ~8.1% (nad svih 25
okruga, 9.1M zapisa), i to neravnomerno (okrug 0: 2.89% za `num_floors`, 24.35% za
`subtype`, ostali red veličine manje). Model bi iz njih učio gustinu mapiranja kao proksi
za urbanost, ne stvarnu izgrađenost.

Uz MLP ide i gradient boosting kao referenca: na tabelarnim podacima ove veličine je jaka
polazna tačka, pa se vidi gde MLP stoji u odnosu na nju.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

fp = pd.read_parquet(config.NASELJE_FOOTPRINTS)

# Spisak atributa je u scripts/config.py (isti izvor iz kog ih racuna per_naselje):
# heavy-tailed brojaci (FP_LOG) idu kroz log1p, ogranicene velicine (FP_LIN) ostaju.
LOG_ATRIBUTI = config.FP_LOG
ATRIBUTI = config.FP_ATRIBUTI

tab = fp[["naselje_maticni_broj", "opstina_maticni_broj", "pop", *ATRIBUTI]].dropna().copy()
for c in LOG_ATRIBUTI:
    tab[c] = np.log1p(tab[c].clip(lower=0))
tab["ylog"] = np.log1p(tab["pop"]).astype("float32")

FOLDS_TAB = make_folds(tab)
print(f"tabelarni skup: {len(tab)} naselja | {tab.opstina_maticni_broj.nunique()} opstina "
      f"| {len(ATRIBUTI)} atributa | foldova {len(FOLDS_TAB)}")
print(f"(raster skup je {len(df)} naselja - tabelarni ne trazi satelitski isecak)")

In [ ]:
CFG_TAB = {
    "hidden": (64, 32),
    "dropout": 0.15,
    "epochs": 200,
    "batch_size": 128,
    "lr": 3e-3,
    "weight_decay": 1e-4,
    "seed": CFG["seed"],
}


class TabMLP(nn.Module):
    # Mali MLP nad standardizovanim atributima; cilj log1p(pop).

    def __init__(self, n_ulaza, hidden, dropout):
        super().__init__()
        slojevi, prethodni = [], n_ulaza
        for h in hidden:
            slojevi += [nn.Linear(prethodni, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prethodni = h
        slojevi.append(nn.Linear(prethodni, 1))
        self.net = nn.Sequential(*slojevi)

    def forward(self, x):
        return self.net(x)


def train_fold_tab(train_frame, val_frame):
    # MLP na jednom foldu; standardizacija se fituje samo na trening delu.
    skaler = StandardScaler().fit(train_frame[ATRIBUTI].values)
    Xtr = torch.tensor(skaler.transform(train_frame[ATRIBUTI].values), dtype=torch.float32)
    Xva = torch.tensor(skaler.transform(val_frame[ATRIBUTI].values), dtype=torch.float32).to(DEVICE)
    ytr = torch.tensor(train_frame["ylog"].values, dtype=torch.float32).unsqueeze(1)
    yva = val_frame["ylog"].values

    dl = DataLoader(TensorDataset(Xtr, ytr), batch_size=CFG_TAB["batch_size"], shuffle=True,
                    generator=torch.Generator().manual_seed(CFG_TAB["seed"]))
    net = TabMLP(len(ATRIBUTI), CFG_TAB["hidden"], CFG_TAB["dropout"]).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=CFG_TAB["lr"],
                            weight_decay=CFG_TAB["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG_TAB["epochs"])
    kriterijum = nn.HuberLoss()

    best_r2, best_pred = -1e9, None
    for e in range(CFG_TAB["epochs"]):
        net.train()
        ukupno = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = kriterijum(net(xb), yb)
            loss.backward()
            opt.step()
            ukupno += loss.item() * len(xb)
        sched.step()
        net.eval()
        with torch.no_grad():
            P = net(Xva).cpu().numpy().ravel()
        r2 = r2_score(yva, P)
        mlflow.log_metrics({"tab_train_loss": ukupno / len(Xtr), "tab_val_r2": r2}, step=e)
        if r2 > best_r2:                       # najbolji po validaciji ovog folda
            best_r2, best_pred = r2, P
    return best_r2, np.clip(np.expm1(best_pred), 0, None)


def train_fold_gbm(train_frame, val_frame):
    # Gradient boosting na istim atributima i istom foldu - referentna linija.
    reg = HistGradientBoostingRegressor(
        max_iter=400, learning_rate=0.05, max_depth=6,
        l2_regularization=1.0, random_state=CFG_TAB["seed"],
    ).fit(train_frame[ATRIBUTI].values, train_frame["ylog"].values)
    P = reg.predict(val_frame[ATRIBUTI].values)
    return r2_score(val_frame["ylog"].values, P), np.clip(np.expm1(P), 0, None)

In [ ]:
def run_tab(ime, fold_fn):
    # Puna GroupKFold CV jednog tabelarnog modela; snima OOF parquet za fuziju.
    oof = pd.Series(np.nan, index=tab.naselje_maticni_broj.values, dtype="float64")
    fold_r2 = []
    with mlflow.start_run(run_name=f"{ime}-cv{len(FOLDS_TAB)}"):
        mlflow.log_params({**CFG_TAB, "pristup": ime, "n_atributa": len(ATRIBUTI),
                           "atributi": ",".join(ATRIBUTI),
                           "cv": f"GroupKFold(opstina) x{len(FOLDS_TAB)}",
                           "n_naselja": len(tab),
                           "n_opstina": int(tab.opstina_maticni_broj.nunique())})
        for fold, (tr, va) in enumerate(FOLDS_TAB):
            with mlflow.start_run(run_name=f"{ime}-fold{fold}", nested=True):
                r2, pred = fold_fn(tr, va)
                mlflow.log_metric("best_val_r2", r2)
                oof.loc[va.naselje_maticni_broj.values] = pred
                fold_r2.append(r2)
                print(f"[{ime} fold {fold}] best val R2 {r2:.3f}")

        oof_pred = oof.loc[tab.naselje_maticni_broj.values].values.astype("float32")
        stvarno = tab["pop"].values.astype("float32")
        agg = run_metrics(fold_r2, stvarno, oof_pred, tab)
        mlflow.log_metrics(agg)
        put_oof = save_oof(tab, oof_pred, ime, OUT_DIR)   # ulaz za fuziju
        mlflow.log_artifact(put_oof)
        fig = cv_summary_figure(fold_r2, agg, stvarno, oof_pred, tab, label=ime)
        plt.show()
        mlflow.log_figure(fig, f"cv_evaluacija_{ime}.png")

    print(summary_line(agg, ime))
    return {"pristup": ime, **agg}, oof_pred


setup_mlflow()   # Databricks workspace; lokalno mlruns/, ili Databricks preko env varijabli
_, oof_tab = run_tab("footprint_tab", train_fold_tab)
_, oof_gbm = run_tab("footprint_gbm", train_fold_gbm)

## Poređenje: da li raster nosi nešto preko brojača?

Pitanje je da li prostorni raspored zgrada, koji vidi samo ResNet nad rasterom, dodaje
informaciju preko golih brojača iz tabelarnih atributa. Zato ista tri modela (tabelarni
MLP, GBM referenca i ResNet nad rasterom) poredimo direktno, na preseku naselja, jer
raster postoji samo tamo gde postoji satelitski isečak, dok tabelarni pokriva sve.

Ako ResNet nad rasterom ne pobedi tabelarni model, prostorni raspored zgrada u naselju ne
nosi informaciju preko one koju već nose brojači, što je nalaz, ne neuspeh.

In [ ]:
# samo OOF metrike: presek se racuna nad vec gotovim predikcijama,
# pa po-fold CV R2 ovde nema smisla (vidi ga u MLflow runu svakog modela)
KOLONE = GLAVNE_KOLONE[1:]   # bez cv_mean_val_r2, videti komentar iznad

# raster OOF se cita iz parqueta koji je run() vec snimio (egzaktno iste
# predikcije na kojima su prijavljene metrike, bez ponovnog forward prolaza)
ras = pd.read_parquet(f"{OUT_DIR}/oof_footprint.parquet").set_index("naselje_maticni_broj")["pred"]

presek = sorted(set(tab.naselje_maticni_broj) & set(ras.index))
p_tab = tab.set_index("naselje_maticni_broj").loc[presek]
p_ras = ras.loc[presek].values
stvarno_p = p_tab["pop"].values.astype("float32")

redovi = []
for ime, oof_pun in [("footprint_tab (MLP)", oof_tab), ("footprint_gbm (GBM)", oof_gbm)]:
    pred_p = pd.Series(oof_pun, index=tab.naselje_maticni_broj.values).loc[presek].values
    redovi.append({"model": ime, **oof_metrics(stvarno_p, pred_p, p_tab.reset_index())})
redovi.append({"model": "footprint (ResNet nad rasterom)",
               **oof_metrics(stvarno_p, p_ras, p_tab.reset_index())})

poredjenje_fp = pd.DataFrame(redovi).set_index("model").reindex(columns=KOLONE)
print(f"presek: {len(presek)} naselja (tabelarni {len(tab)}, raster {len(ras)})")
print("\n=== Pristup 2: strukturirani atributi vs raster ===")
display(poredjenje_fp)